In [21]:
import sys
sys.path.append("../src")

import importlib
import pandas as pd

import metrics
import backtesting

importlib.reload(metrics)
importlib.reload(backtesting)

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01

prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

In [22]:
def run_robustness_returns(window=WINDOW, step=STEP, vol_target=VOL_TARGET,
                           max_leverage=MAX_LEVERAGE,
                           transaction_cost=TRANSACTION_COST,
                           funding_spread=FUNDING_SPREAD):

    portfolio_returns = {}

    for name, (strategy, strategy_groups) in strategies.items():
        port_ret, _, _, _, _ = backtesting.backtest(
            returns,
            window=window,
            step=step,
            allocation_func=strategy,
            groups=strategy_groups,
            vol_target=vol_target,
            max_leverage=max_leverage,
            risk_free_returns=risk_free_returns,
            transaction_cost=transaction_cost,
            funding_spread=funding_spread
        )

        portfolio_returns[name] = port_ret

    return pd.DataFrame(portfolio_returns)

In [23]:
# Window

windows = {"2Y": 2 * 252, "3Y": 3 * 252,"5Y": 5 * 252}

window_returns = {}

for label, window in windows.items():
    window_returns[label] = run_robustness_returns(window=window)

common_index = (
    window_returns["2Y"].index
    .intersection(window_returns["3Y"].index)
    .intersection(window_returns["5Y"].index)
)

for label, port_returns in window_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

2Y


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.064154,0.104350,0.504386,-0.248580
1,Inverse Vol,0.048901,0.080594,0.446425,-0.248562
2,Max Geo,0.074855,0.110828,0.571490,-0.262939
3,Max Sharpe,0.065416,0.090832,0.577896,-0.206872
4,Carver,0.037092,0.061082,0.380911,-0.217831
5,Hierarchical Inv Vol,0.040507,0.065413,0.410166,-0.223019
6,SPY,0.083518,0.105533,0.670693,-0.217877
7,60/40,0.092647,0.105191,0.752253,-0.210350
8,Tech,0.100119,0.103394,0.829485,-0.149707


3Y


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.066732,0.099936,0.546446,-0.270695
1,Inverse Vol,0.050142,0.077598,0.475863,-0.239383
2,Max Geo,0.093019,0.102307,0.773997,-0.222613
3,Max Sharpe,0.074405,0.086549,0.699372,-0.225831
4,Carver,0.038681,0.061294,0.404805,-0.212516
5,Hierarchical Inv Vol,0.041735,0.065341,0.428619,-0.215909
6,SPY,0.086331,0.099561,0.730864,-0.244055
7,60/40,0.094442,0.100017,0.802322,-0.234723
8,Tech,0.101255,0.099649,0.867223,-0.161984


5Y


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.065321,0.094608,0.557689,-0.252080
1,Inverse Vol,0.051387,0.077152,0.493495,-0.243406
2,Max Geo,0.080025,0.099595,0.672221,-0.217016
3,Max Sharpe,0.071013,0.086186,0.665261,-0.269880
4,Carver,0.037443,0.060351,0.390379,-0.214199
5,Hierarchical Inv Vol,0.040907,0.064948,0.418549,-0.218017
6,SPY,0.083466,0.096807,0.721574,-0.245975
7,60/40,0.091292,0.098468,0.784112,-0.235671
8,Tech,0.100112,0.099223,0.860055,-0.167825


In [24]:
# Step

steps = {"Weekly": 5,"Monthly": 21, "Quarterly": 63}

step_returns = {}

for label, step in steps.items():
    step_returns[label] = run_robustness_returns(step=step)

common_index = (
    step_returns["Weekly"].index
    .intersection(step_returns["Monthly"].index)
    .intersection(step_returns["Quarterly"].index)
)

for label, port_returns in step_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

Weekly


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.051116,0.108007,0.384203,-0.300454
1,Inverse Vol,0.040365,0.090668,0.324984,-0.295555
2,Max Geo,0.076868,0.101655,0.639687,-0.248624
3,Max Sharpe,0.063843,0.082392,0.620152,-0.217169
4,Carver,0.037594,0.068071,0.367282,-0.212750
5,Hierarchical Inv Vol,0.039868,0.073724,0.374264,-0.216092
6,SPY,0.065144,0.109601,0.501024,-0.309345
7,60/40,0.072502,0.110148,0.561551,-0.314352
8,Tech,0.084096,0.106570,0.677684,-0.269134


Monthly


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.052347,0.108601,0.393435,-0.305042
1,Inverse Vol,0.040634,0.090069,0.329396,-0.297302
2,Max Geo,0.080331,0.105696,0.649546,-0.240329
3,Max Sharpe,0.067200,0.083973,0.647564,-0.225831
4,Carver,0.037625,0.067289,0.371213,-0.212516
5,Hierarchical Inv Vol,0.039974,0.072986,0.378711,-0.215909
6,SPY,0.066261,0.110605,0.506927,-0.309052
7,60/40,0.074121,0.111122,0.571137,-0.310480
8,Tech,0.084668,0.106415,0.683462,-0.269885


Quarterly


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.052049,0.111750,0.382990,-0.315912
1,Inverse Vol,0.040961,0.090664,0.331282,-0.301036
2,Max Geo,0.079334,0.106829,0.635158,-0.240647
3,Max Sharpe,0.066885,0.088609,0.614815,-0.219288
4,Carver,0.037060,0.066883,0.364919,-0.214160
5,Hierarchical Inv Vol,0.039382,0.072587,0.372542,-0.217580
6,SPY,0.065059,0.114549,0.483536,-0.315087
7,60/40,0.074417,0.114034,0.561868,-0.317204
8,Tech,0.084063,0.108200,0.668813,-0.284346


In [25]:
# Vol target

vol_targets = {"5%": 0.05,"10%": 0.10,"15%": 0.15}

vol_target_returns = {}

for label, vol_target in vol_targets.items():
    vol_target_returns[label] = run_robustness_returns(vol_target=vol_target)

common_index = (
    vol_target_returns["5%"].index
    .intersection(vol_target_returns["10%"].index)
    .intersection(vol_target_returns["15%"].index)
)

for label, port_returns in vol_target_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

5%


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.034338,0.053526,0.391975,-0.160635
1,Inverse Vol,0.034613,0.054686,0.389621,-0.156325
2,Max Geo,0.047718,0.052416,0.644423,-0.123357
3,Max Sharpe,0.046331,0.049939,0.647520,-0.145096
4,Carver,0.030594,0.050083,0.342844,-0.178949
5,Hierarchical Inv Vol,0.032133,0.051959,0.361047,-0.168300
6,SPY,0.040977,0.054479,0.503468,-0.162774
7,60/40,0.045629,0.054687,0.583246,-0.162579
8,Tech,0.050038,0.052816,0.681746,-0.140162


10%


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.052347,0.108601,0.393435,-0.305042
1,Inverse Vol,0.040634,0.090069,0.329396,-0.297302
2,Max Geo,0.080331,0.105696,0.649546,-0.240329
3,Max Sharpe,0.067200,0.083973,0.647564,-0.225831
4,Carver,0.037625,0.067289,0.371213,-0.212516
5,Hierarchical Inv Vol,0.039974,0.072986,0.378711,-0.215909
6,SPY,0.066261,0.110605,0.506927,-0.309052
7,60/40,0.074121,0.111122,0.571137,-0.310480
8,Tech,0.084668,0.106415,0.683462,-0.269885


15%


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.066312,0.165730,0.385019,-0.429476
1,Inverse Vol,0.048994,0.099857,0.386551,-0.299001
2,Max Geo,0.111421,0.160079,0.651443,-0.345412
3,Max Sharpe,0.080567,0.112937,0.617075,-0.285539
4,Carver,0.037931,0.067720,0.373645,-0.212516
5,Hierarchical Inv Vol,0.040889,0.073994,0.386454,-0.215909
6,SPY,0.089359,0.168795,0.507400,-0.434323
7,60/40,0.098479,0.166653,0.561694,-0.438826
8,Tech,0.117933,0.160964,0.684958,-0.384617


In [26]:
# Max leverage

leverage_caps = {"1x": 1.0,"1.5x": 1.5,"2x": 2.0}

leverage_returns = {}

for label, max_leverage in leverage_caps.items():
    leverage_returns[label] = run_robustness_returns(max_leverage=max_leverage)

common_index = (
    leverage_returns["1.0x"].index
    .intersection(leverage_returns["1.5x"].index)
    .intersection(leverage_returns["2.0x"].index)
)

for label, port_returns in leverage_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

KeyError: '1.0x'